In [1]:
import numpy as np
import pandas as pd
import pickle

INPUT = "code/dataset/Original/global/global-genre_network-2019.csv"
df = pd.read_csv(INPUT, sep="\t")


genres = pd.unique(df[["source", "target"]].values.ravel())
genre_to_idx = {g: i for i, g in enumerate(genres)}
n = len(genres)

i = df["source"].map(genre_to_idx).to_numpy()
j = df["target"].map(genre_to_idx).to_numpy()
w = df["weight"].to_numpy()

adj_numpy = np.zeros((n, n), dtype=np.float32)

for a, b, val in zip(i, j, w):
    adj_numpy[a, b] = max(adj_numpy[a, b], val)
    adj_numpy[b, a] = max(adj_numpy[b, a], val)

#check properties
is_symmetric = np.allclose(adj_numpy, adj_numpy.T)
density = np.count_nonzero(adj_numpy) / adj_numpy.size

print("Symmetric:", is_symmetric)
print("Density:", density)


np.save("my_graph.npy", adj_numpy)
pickle.dump(genres.tolist(), open("matrix_vocab.pkl", "wb"))

Symmetric: True
Density: 0.06016649323621228


In [5]:
print("Largest CC size:", len(largest_cc))

Largest CC size: 294


In [6]:
import numpy as np
import pandas as pd
import pickle
import networkx as nx

INPUT = "code/dataset/Original/global/global-genre_network-2019.csv"
df = pd.read_csv(INPUT, sep="\t")
df["weight"] = df["weight"].astype(float)

genres = pd.unique(df[["source", "target"]].values.ravel())
genre_to_idx = {g: i for i, g in enumerate(genres)}
n = len(genres)

i = df["source"].map(genre_to_idx).to_numpy()
j = df["target"].map(genre_to_idx).to_numpy()
w = df["weight"].to_numpy()

adj_numpy = np.zeros((n, n), dtype=np.float32)

for a, b, val in zip(i, j, w):
    adj_numpy[a, b] = max(adj_numpy[a, b], val)
    adj_numpy[b, a] = max(adj_numpy[b, a], val)

print("Original shape:", adj_numpy.shape)

G = nx.from_numpy_array(adj_numpy)
components = list(nx.connected_components(G))
print("Number of connected components:", len(components))

largest_cc = max(components, key=len)
largest_cc = sorted(largest_cc)
print("Largest CC size:", len(largest_cc))

adj_lcc = adj_numpy[np.ix_(largest_cc, largest_cc)]
genres_lcc = [genres[idx] for idx in largest_cc]

G_lcc = nx.from_numpy_array(adj_lcc)
print("LCC shape:", adj_lcc.shape)
print("LCC connected:", nx.is_connected(G_lcc))

np.save("my_graph.npy", adj_lcc)
pickle.dump(genres_lcc, open("matrix_vocab.pkl", "wb"))

print("Saved my_graph.npy and matrix_vocab.pkl")

Original shape: (310, 310)
Number of connected components: 4
Largest CC size: 294
LCC shape: (294, 294)
LCC connected: True
Saved my_graph.npy and matrix_vocab.pkl
